# BERT 情緒分類：Colab 初學者實作

這份 notebook 讓你在 Google Colab 用 GPU 微調 BERT，判斷推文的情緒是 negative、neutral 或 positive。

你會完成四件事：安裝套件、下載資料與預訓練模型、訓練模型、在 test split 計算 accuracy。完成後會產生 `bert-inf.out` 與 `bert-inf.err`，可用於繳交。

## 開始前：開啟 GPU

在 Colab 上方選單選擇 **Runtime → Change runtime type → T4 GPU**（介面文字可能略有不同），再按 Save。若沒有 GPU 配額，也可以先用 CPU 跑完流程，但訓練會慢很多。

依序執行每個程式區塊：點區塊左側的播放按鈕，或按 `Shift + Enter`。第一次執行會下載模型和資料集，因此比後續執行久。

In [ ]:
# 有看到 GPU 名稱，表示 Colab 已經配置 GPU。
!nvidia-smi


## 1. 下載課程程式與安裝套件

Colab 每次重新連線都可能是新的機器，所以需要重新下載專案與套件。這個步驟不會修改 GitHub 上的專案。

In [ ]:
!git clone https://github.com/NTHU-SC/BERT-Finetune-HPCAI-Camp-2026.git
%cd /content/BERT-Finetune-HPCAI-Camp-2026
!pip -q install transformers datasets scikit-learn evaluate accelerate


In [ ]:
import torch

print('PyTorch version:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('請回到 Runtime 設定開啟 GPU；沒有 GPU 也能跑，但會比較慢。')


## 2. 設定這次實驗

先用下面較小的設定完成一次流程。`train_samples` 是訓練資料量；`epochs` 是完整看過訓練資料的次數；`learning_rate` 決定每次更新的步伐；`batch_size` 太大可能造成 GPU 記憶體不足。

第一次成功後，再一次只改一到兩個參數，並且使用不同的 `output_dir`，才能知道是哪個改動帶來差異。

In [ ]:
# 初學者的安全起點：完成一次訓練大約需要數分鐘，時間依 Colab GPU 而不同。
model_name = 'google-bert/bert-base-uncased'
output_dir = './colab_output'
train_samples = 5000
eval_samples = 500
batch_size = 32
epochs = 3
learning_rate = 3e-5
seed = 42


## 3. 微調 BERT

下一格會執行 `Train.py`。程式會下載 `mteb/tweet_sentiment_extraction`，用訓練資料微調 BERT，並在每個 epoch 後印出 `eval_accuracy`。

請注意：`eval_accuracy` 是保留 validation split 的結果，用來幫助你選設定；最後的成績要看下一步 test split 的 accuracy。

In [ ]:
!python Train.py \
  --model {model_name} \
  --output {output_dir} \
  --train-samples {train_samples} \
  --eval-samples {eval_samples} \
  --batch-size {batch_size} \
  --epochs {epochs} \
  --learning-rate {learning_rate} \
  --seed {seed} 2>&1 | tee bert-train.out


訓練成功後，模型會存在 `colab_output/checkpoint-final`。下面先確認 checkpoint 是否存在，再看訓練輸出的最後幾行。

In [ ]:
!ls {output_dir}/checkpoint-final
!tail -n 30 bert-train.out


## 4. 在 test split 計算 accuracy

現在使用剛才的 checkpoint 執行 `Inference.py`。這一步不會繼續訓練，而是對完整 test split 做預測並計算 accuracy。輸出會分別寫入 `bert-inf.out` 與 `bert-inf.err`。

In [ ]:
!python Inference.py --model {output_dir}/checkpoint-final > bert-inf.out 2> bert-inf.err
!cat bert-inf.out
!cat bert-inf.err


## 5. 下載結果並繳交

下面會把兩個檔案下載到你的電腦。連同實驗報告的 HackMD 連結，透過課程的 Google Form 繳交。

In [ ]:
from google.colab import files

files.download('bert-inf.out')
files.download('bert-inf.err')


## 下一步可以嘗試什麼？

1. 先記錄這次的參數、`eval_accuracy`、test accuracy 與訓練時間。
2. 修改上一個「設定這次實驗」區塊的一到兩個參數。
3. 把 `output_dir` 換成新名稱，例如 `./experiment-bs64`，避免覆蓋舊模型。
4. 重新執行訓練、推論與下載區塊。

若出現 `CUDA out of memory`，先把 `batch_size` 從 32 降為 16 或 8。請勿修改 `Inference.py`，也不要更換模型、資料集或資料內容。